# Great Expectations (GX) – Silver suite (strict validation)

Cilj: validirati SILVER sloj (očšćeni podaci) prije GOLD sloja.

Dataset/tablica:
- Project: tpiuo-labosi
- Dataset: stackoverflow_dbt
- Table: stackoverflow_questions_silver

In [9]:
import os
import json
from datetime import datetime, timezone

from google.cloud import bigquery
import great_expectations as gx

PROJECT_ID = "tpiuo-labosi"
DBT_DATASET = "stackoverflow_dbt"
SILVER_TABLE = f"{PROJECT_ID}.{DBT_DATASET}.stackoverflow_questions_silver"

bq = bigquery.Client(project=PROJECT_ID)
silver_df = bq.query(f"SELECT * FROM `{SILVER_TABLE}`").to_dataframe()
silver_df.head(), silver_df.shape

(   question_id                                        title_clean  \
 0     79855479  Flow Layout with customtkinter adds extra padd...   
 1     79855549                    pyttsx3 skips first few letters   
 2     79855374  Fatal error postgres password authentication f...   
 3     79855536  How does attribute access actually work intern...   
 4     79855386  Access ir rule res partner Odoo V19 Troublesho...   
 
                                                 link  \
 0  https://stackoverflow.com/questions/79855479/f...   
 1  https://stackoverflow.com/questions/79855549/p...   
 2  https://stackoverflow.com/questions/79855374/f...   
 3  https://stackoverflow.com/questions/79855536/h...   
 4  https://stackoverflow.com/questions/79855386/a...   
 
                  created_ts created_date          last_activity_ts  \
 0 2025-12-26 18:43:42+00:00   2025-12-26 2026-01-10 16:28:22+00:00   
 1 2025-12-26 22:16:31+00:00   2025-12-26 2025-12-28 07:39:40+00:00   
 2 2025-12-26 15:05:4

In [10]:
context = gx.get_context(mode="file")

try:
    ds = context.data_sources.get("so_pandas")
except Exception:
    ds = context.data_sources.add_pandas(name="so_pandas")

try:
    asset = ds.get_asset("silver_questions")
except Exception:
    asset = ds.add_dataframe_asset(name="silver_questions")

try:
    silver_batch_def = asset.get_batch_definition("whole_df_silver")
except Exception:
    silver_batch_def = asset.add_batch_definition_whole_dataframe("whole_df_silver")

silver_batch_def


BatchDefinition(id='b672ea23-2d93-47ab-9a00-5be792368b00', name='whole_df_silver', partitioner=None)

In [11]:
suite_name = "stackoverflow_silver_suite"

try:
    silver_suite = context.suites.get(suite_name)
except Exception:
    silver_suite = gx.ExpectationSuite(name=suite_name)
    silver_suite = context.suites.add(silver_suite)

ex = gx.expectations

# id unique + not null
silver_suite.add_expectation(ex.ExpectColumnToExist(column="question_id"))
silver_suite.add_expectation(ex.ExpectColumnValuesToNotBeNull(column="question_id"))
silver_suite.add_expectation(ex.ExpectColumnValuesToBeUnique(column="question_id"))

# sentiment constraints
silver_suite.add_expectation(ex.ExpectColumnToExist(column="sentiment"))
silver_suite.add_expectation(ex.ExpectColumnValuesToNotBeNull(column="sentiment"))
silver_suite.add_expectation(
    ex.ExpectColumnValuesToBeInSet(
        column="sentiment",
        value_set=["HIGH_ENGAGEMENT", "POSITIVE", "NEGATIVE", "NEUTRAL"],
    )
)

# title length constraint (per instructions-style example)
silver_suite.add_expectation(ex.ExpectColumnToExist(column="title_clean"))
silver_suite.add_expectation(ex.ExpectColumnValuesToNotBeNull(column="title_clean"))

silver_suite

{
  "name": "stackoverflow_silver_suite",
  "id": "f03631b4-992a-491d-ab29-57410c02c896",
  "expectations": [
    {
      "type": "expect_column_to_exist",
      "kwargs": {
        "column": "question_id"
      },
      "meta": {},
      "id": "032e13e3-ab2c-4214-9616-04b6eb8182dc",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "question_id"
      },
      "meta": {},
      "id": "b9d74ed7-fea8-4e1a-9ff7-742ae01ea389",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_unique",
      "kwargs": {
        "column": "question_id"
      },
      "meta": {},
      "id": "49e1982e-28f9-4f35-bb40-3f550797e9cd",
      "severity": "critical"
    },
    {
      "type": "expect_column_to_exist",
      "kwargs": {
        "column": "sentiment"
      },
      "meta": {},
      "id": "cf97db09-7a78-44f0-a85f-3b0725d5c1f7",
      "severity": "critical"
    },
    {
      "type": "exp

In [12]:
ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
validation_name = f"silver_validation_{ts}"

vdef = gx.ValidationDefinition(data=silver_batch_def, suite=silver_suite, name=validation_name)
silver_result = vdef.run(batch_parameters={"dataframe": silver_df})

os.makedirs("gx_results", exist_ok=True)
out_path = f"gx_results/{validation_name}.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(silver_result.to_json_dict(), f, ensure_ascii=False, indent=2)

silver_result.success, out_path

Calculating Metrics: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 33/33 [00:00<00:00, 811.18it/s]


(True, 'gx_results/silver_validation_20260120T175508Z.json')